In [ ]:
# 让 notebook 无论从哪个目录启动都能 import retailinsight
import sys
from pathlib import Path

_candidates = [Path.cwd(), *Path.cwd().parents]
_root = next(p for p in _candidates if (p / "retailinsight" / "__init__.py").exists())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print("项目根目录:", _root)


# 04 · 关联规则挖掘
> 目标：把「一起买」的商品组合变成可执行的陈列与捆绑策略。

In [ ]:
import pandas as pd

from retailinsight.io.loader import read_parquet
from retailinsight.settings import MART_RULES

rules = read_parquet(MART_RULES)
print(f"共 {len(rules)} 条规则（min_support 1% / min_confidence 30% / lift > 1.5）")
rules.head(10)


## 4.1 规则强度分布

In [ ]:
print("lift 分布:", rules["lift"].describe().round(2).to_dict())
print("\nconfidence 分布:", rules["confidence"].describe().round(2).to_dict())
rules["lift_bin"] = pd.cut(rules["lift"], [1.5, 2, 3, 5, 10], right=False)
rules.groupby("lift_bin", observed=True)["lift"].size()


## 4.2 Top 20 强关联规则

In [ ]:
rules.nlargest(20, "lift")[["antecedents", "consequents", "support",
                                  "confidence", "lift"]].round(4)


## 4.3 埋点回收验证：4 组预埋组合

In [ ]:
from retailinsight.io.schema import ANCHOR_COMBOS

for a, b, _target in ANCHOR_COMBOS:
    hit = rules[(rules["antecedents"].astype(str).str.contains(a, regex=False))
                & (rules["consequents"].astype(str).str.contains(b, regex=False))]
    print(f"  {a} → {b}: {'✅ lift=' + format(hit['lift'].max(), '.2f') if len(hit) else '❌ 未回收'}")


## 4.4 高频共同出现品类对

In [ ]:
# 前后件都可能是多品类组合，逐行展开成「品类对」
_rows = []
for _, r in rules.iterrows():
    for a in str(r["antecedents"]).split("/"):
        for b in str(r["consequents"]).split("/"):
            if a != b:
                _rows.append({"a": a, "b": b, "lift": float(r["lift"]),
                              "confidence": float(r["confidence"])})
_pair_df = pd.DataFrame(_rows)

pairs = (_pair_df.groupby(["a", "b"])
         .agg(n=("lift", "size"), lift=("lift", "max"), confidence=("confidence", "max"))
         .sort_values(["n", "lift"], ascending=False))
pairs.head(15)
